# Silver Transformation — Data Cleaning & Quality Validation

## Purpose

This notebook transforms raw Policies and Claims data from the Bronze layer into clean, standardized, and validated Delta tables in the Silver layer.

The main objective is to ensure data quality, consistency, and referential integrity before performing business analysis and KPI calculations.

## Why is this important?

- **Data Quality:** Identify missing values, duplicate records, and invalid business data.
- **Data Standardization:** Convert policy identifiers to consistent data types and apply meaningful column names.
- **Business Rule Validation:** Verify policy exposure, claim counts, driver ages, and claim amounts.
- **Referential Integrity:** Identify claim records without a matching policy.
- **Data Traceability:** Preserve ingestion metadata for auditing and troubleshooting.
- **Data Reliability:** Separate valid claims from orphan claims without losing source records.

## Expected Outcome

The notebook creates three Delta tables:

- `auto_insurance.silver.policies` — Clean and standardized policy records.
- `auto_insurance.silver.claims` — Valid claim records linked to existing policies.
- `auto_insurance.silver.claims_orphan` — Unmatched claim records preserved for investigation.

These tables provide a reliable foundation for claims reconciliation and downstream Gold-layer analytics.

**Note:** This notebook processes a complete data snapshot using overwrite mode. Incremental processing and automated data quality monitoring will be implemented in future iterations.

In [0]:
# Import PySpark functions for data profiling and validation
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define source tables in the Bronze layer
policies_table = "auto_insurance.bronze.policies_raw"
claims_table = "auto_insurance.bronze.claims_raw"

# Read raw policies and claims data from Delta tables
df_policies = spark.table(policies_table)
df_claims = spark.table(claims_table)

# Validate the number of records loaded from Bronze
print("Policies count:", df_policies.count())
print("Claims count:", df_claims.count())

# Inspect the schemas of both datasets
df_policies.printSchema()
df_claims.printSchema()

Policies count: 678013
Claims count: 26639
root
 |-- IDpol: double (nullable = true)
 |-- ClaimNb: integer (nullable = true)
 |-- Exposure: double (nullable = true)
 |-- VehPower: integer (nullable = true)
 |-- VehAge: integer (nullable = true)
 |-- DrivAge: integer (nullable = true)
 |-- BonusMalus: integer (nullable = true)
 |-- VehBrand: string (nullable = true)
 |-- VehGas: string (nullable = true)
 |-- Area: string (nullable = true)
 |-- Density: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)

root
 |-- IDpol: integer (nullable = true)
 |-- ClaimAmount: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [0]:
# Calculate the number of NULL values for each column in Policies
policies_null_counts = df_policies.select([
    F.count(
        F.when(F.col(column).isNull(), 1)
    ).alias(column)
    for column in df_policies.columns
])

# Display missing value statistics for Policies
display(policies_null_counts)

# Calculate the number of NULL values for each column in Claims
claims_null_counts = df_claims.select([
    F.count(
        F.when(F.col(column).isNull(), 1)
    ).alias(column)
    for column in df_claims.columns
])

# Display missing value statistics for Claims
display(claims_null_counts)

IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region,ingestion_timestamp,source_file,batch_id
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


IDpol,ClaimAmount,ingestion_timestamp,source_file,batch_id
0,0,0,0,0


In [0]:
# Identify policy IDs that appear more than once
duplicate_policies = (
    df_policies
    .groupBy("IDpol")
    .count()
    .filter(F.col("count") > 1)
)

# Count the number of policy IDs with duplicate records
duplicate_policy_count = duplicate_policies.count()

print("Duplicate policy IDs:", duplicate_policy_count)

# Display a sample of duplicated policy IDs
display(duplicate_policies.limit(10))

# Note: Multiple claims per policy are valid.
# Therefore, duplicate IDpol values in Claims are not treated as errors.

Duplicate policy IDs: 0


IDpol,count


In [0]:
# Check business rules related to policy exposure,
# claim frequency, driver age, and policy identifiers
policies_quality_checks = df_policies.select(
    F.count(
        F.when(F.col("Exposure") <= 0, 1)
    ).alias("invalid_exposure"),

    F.count(
        F.when(F.col("ClaimNb") < 0, 1)
    ).alias("negative_claim_count"),

    F.count(
        F.when(F.col("DrivAge") < 18, 1)
    ).alias("underage_drivers"),

    F.count(
        F.when(F.col("IDpol").isNull(), 1)
    ).alias("missing_policy_id")
)

# Display policy data quality results
display(policies_quality_checks)

invalid_exposure,negative_claim_count,underage_drivers,missing_policy_id
0,0,0,0


In [0]:
# Check for invalid claim amounts and missing policy identifiers
claims_quality_checks = df_claims.select(
    F.count(
        F.when(F.col("ClaimAmount") <= 0, 1)
    ).alias("invalid_claim_amount"),

    F.count(
        F.when(F.col("ClaimAmount").isNull(), 1)
    ).alias("missing_claim_amount"),

    F.count(
        F.when(F.col("IDpol").isNull(), 1)
    ).alias("missing_policy_id")
)

# Display claim data quality results
display(claims_quality_checks)

invalid_claim_amount,missing_claim_amount,missing_policy_id
0,0,0


In [0]:
# Identify policy IDs containing fractional values
# before converting them from DOUBLE to BIGINT
fractional_policy_ids = df_policies.filter(
    F.col("IDpol").isNotNull() &
    (F.col("IDpol") != F.floor(F.col("IDpol")))
)

# Count policy IDs that cannot be safely converted to integers
fractional_policy_id_count = fractional_policy_ids.count()

print("Fractional policy IDs:", fractional_policy_id_count)

# Display a sample of invalid identifiers
display(fractional_policy_ids.limit(10))

Fractional policy IDs: 0


IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region,ingestion_timestamp,source_file,batch_id


In [0]:
# Extract distinct policy identifiers from the Policies dataset
policy_ids = (
    df_policies
    .select(F.col("IDpol").cast("long").alias("IDpol"))
    .distinct()
)

# Standardize claim policy identifiers for comparison
claim_ids = df_claims.select(
    F.col("IDpol").cast("long").alias("IDpol")
)

# Identify claims referencing policy IDs not found in Policies
orphan_claims = claim_ids.join(
    policy_ids,
    on="IDpol",
    how="left_anti"
)

# Count claim records without a matching policy
orphan_claim_count = orphan_claims.count()

print("Orphan claims:", orphan_claim_count)

# Display a sample of unmatched policy identifiers
display(orphan_claims.limit(10))

Orphan claims: 195


IDpol
2262511
2262511
2262511
2262511
2262511
2262511
2262511
2262511
2262511
2262511


In [0]:
# Collect key profiling metrics into a summary dictionary
profiling_summary = {
    "policies_count": df_policies.count(),
    "claims_count": df_claims.count(),
    "duplicate_policy_ids": duplicate_policy_count,
    "fractional_policy_ids": fractional_policy_id_count,
    "orphan_claims": orphan_claim_count
}

# Convert profiling metrics into a Spark DataFrame
summary_data = [
    (metric, int(value))
    for metric, value in profiling_summary.items()
]

df_profiling_summary = spark.createDataFrame(
    summary_data,
    ["metric", "value"]
)

# Display the final profiling summary
display(df_profiling_summary)

metric,value
policies_count,678013
claims_count,26639
duplicate_policy_ids,0
fractional_policy_ids,0
orphan_claims,195


next step

In [0]:
# Standardize policy identifiers to BIGINT across both datasets
# This ensures consistent data types for joins and downstream transformations

df_policies_standardized = (
    df_policies
    .withColumn("IDpol", F.col("IDpol").cast("long"))
)

df_claims_standardized = (
    df_claims
    .withColumn("IDpol", F.col("IDpol").cast("long"))
)

# Verify standardized data types
df_policies_standardized.printSchema()
df_claims_standardized.printSchema()

root
 |-- IDpol: long (nullable = true)
 |-- ClaimNb: integer (nullable = true)
 |-- Exposure: double (nullable = true)
 |-- VehPower: integer (nullable = true)
 |-- VehAge: integer (nullable = true)
 |-- DrivAge: integer (nullable = true)
 |-- BonusMalus: integer (nullable = true)
 |-- VehBrand: string (nullable = true)
 |-- VehGas: string (nullable = true)
 |-- Area: string (nullable = true)
 |-- Density: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)

root
 |-- IDpol: long (nullable = true)
 |-- ClaimAmount: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [0]:
# Prepare validated policy records for the Silver layer
# Preserve source attributes and ingestion metadata for traceability

df_silver_policies = (
    df_policies_standardized
    .withColumnRenamed("IDpol", "policy_id")
    .withColumnRenamed("ClaimNb", "claim_count")
    .withColumnRenamed("Exposure", "exposure")
    .withColumnRenamed("VehPower", "vehicle_power")
    .withColumnRenamed("VehAge", "vehicle_age")
    .withColumnRenamed("DrivAge", "driver_age")
    .withColumnRenamed("BonusMalus", "bonus_malus")
    .withColumnRenamed("VehBrand", "vehicle_brand")
    .withColumnRenamed("VehGas", "vehicle_fuel")
    .withColumnRenamed("Area", "area")
    .withColumnRenamed("Density", "population_density")
    .withColumnRenamed("Region", "region")
)

# Preview standardized policy records
display(df_silver_policies.limit(10))

policy_id,claim_count,exposure,vehicle_power,vehicle_age,driver_age,bonus_malus,vehicle_brand,vehicle_fuel,area,population_density,region,ingestion_timestamp,source_file,batch_id
1,1,0.1,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
3,1,0.77,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
5,1,0.75,6,2,52,50,B12,Diesel,B,54,Picardie,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
10,1,0.09,7,0,46,50,B12,Diesel,B,76,Aquitaine,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
11,1,0.84,7,0,46,50,B12,Diesel,B,76,Aquitaine,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
13,1,0.52,6,2,38,50,B12,Regular,E,3003,Nord-Pas-de-Calais,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
15,1,0.45,6,2,38,50,B12,Regular,E,3003,Nord-Pas-de-Calais,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
17,1,0.27,7,0,33,68,B12,Diesel,C,137,Languedoc-Roussillon,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
18,1,0.71,7,0,33,68,B12,Diesel,C,137,Languedoc-Roussillon,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a
21,1,0.15,7,0,41,50,B12,Diesel,B,60,Pays-de-la-Loire,2026-10-14T08:36:25.837Z,freMTPL2freq.csv,70fa346a-6af7-4796-a902-71aa3cd9bc9a


In [0]:
# Extract unique policy identifiers for referential integrity validation
df_policy_keys = (
    df_silver_policies
    .select("policy_id")
    .distinct()
)

# Standardize claim column names
df_claims_prepared = (
    df_claims_standardized
    .withColumnRenamed("IDpol", "policy_id")
    .withColumnRenamed("ClaimAmount", "claim_amount")
)

# Identify claims with matching policy records
df_silver_claims = (
    df_claims_prepared
    .join(df_policy_keys, on="policy_id", how="left_semi")
)

# Identify claims without matching policy records
df_orphan_claims = (
    df_claims_prepared
    .join(df_policy_keys, on="policy_id", how="left_anti")
    .withColumn(
        "quarantine_reason",
        F.lit("POLICY_NOT_FOUND")
    )
)

# Validate claim distribution
print("Valid claims:", df_silver_claims.count())
print("Orphan claims:", df_orphan_claims.count())

Valid claims: 26444
Orphan claims: 195


In [0]:
# Persist standardized policy records as a managed Delta table
(
    df_silver_policies.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("auto_insurance.silver.policies")
)

# Persist valid claims with matching policy identifiers
(
    df_silver_claims.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("auto_insurance.silver.claims")
)

# Preserve orphan claims for investigation and future reprocessing
(
    df_orphan_claims.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("auto_insurance.silver.claims_orphan")
)

print("Silver Delta tables created successfully.")

Silver Delta tables created successfully.


In [0]:
# Read persisted Silver tables
df_policies_silver = spark.table(
    "auto_insurance.silver.policies"
)

df_claims_silver = spark.table(
    "auto_insurance.silver.claims"
)

df_orphans_silver = spark.table(
    "auto_insurance.silver.claims_orphan"
)

# Validate record counts and claim reconciliation
policies_count = df_policies_silver.count()
valid_claims_count = df_claims_silver.count()
orphan_claims_count = df_orphans_silver.count()
bronze_claims_count = df_claims.count()

print("Silver policies:", policies_count)
print("Silver valid claims:", valid_claims_count)
print("Silver orphan claims:", orphan_claims_count)

# Ensure that no claim records were lost during transformation
assert valid_claims_count + orphan_claims_count == bronze_claims_count, (
    "Claims reconciliation failed: Silver counts do not match Bronze."
)

# Ensure that policy records were preserved
assert policies_count == df_policies.count(), (
    "Policies reconciliation failed: Silver count does not match Bronze."
)

print("Silver reconciliation completed successfully.")

Silver policies: 678013
Silver valid claims: 26444
Silver orphan claims: 195
Silver reconciliation completed successfully.
